# LoRA fine-tuning: a small generator for the ticket RAG system

Plain Hugging Face Transformers + PEFT (no LangChain, no TRL).

**Before running**
1. Runtime > Change runtime type > select a GPU (T4 is fine).
2. Upload `train.jsonl` and `test.jsonl` (from `lora-finetuning/data/`) to `/content` via the Files panel.
3. For the upload step: add a Colab Secret named `HF_TOKEN` (key icon, left sidebar) with your Hugging Face *write* token, and turn on "Notebook access".

In [1]:
!pip install -q -U transformers peft datasets accelerate
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
# If Colab asks to restart the session after the install: Runtime > Restart session, then run again.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 58.1 MB/s eta 0:00:00
CUDA available: True
Tesla T4


In [2]:
# ===== Config =====
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # any small chat model on the Hub; check the Hub for newer small Qwen models
MAX_LEN = 768
SEED = 42

EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 8
GRAD_ACC = 2

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

USE_DRIVE = True    # save outputs to Google Drive so they survive a disconnected session
HF_REPO_NAME = "qwen2.5-0.5b-ticket-rag-lora"

TRAIN_PATH = "/content/train.jsonl"
TEST_PATH = "/content/test.jsonl"

In [3]:
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/lora_ticket_rag")
else:
    OUTPUT_DIR = Path("/content/lora_ticket_rag")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR)

Mounted at /content/drive
Output dir: /content/drive/MyDrive/lora_ticket_rag


In [4]:
import json, os, random
import numpy as np

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

for p in (TRAIN_PATH, TEST_PATH):
    assert os.path.exists(p), f"{p} not found. Upload it via the Files panel (left sidebar) first."

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_rows = read_jsonl(TRAIN_PATH)
test_rows = read_jsonl(TEST_PATH)
print("train:", len(train_rows), "| test:", len(test_rows))

train: 3432 | test: 452


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# fp32 weights + fp16 mixed precision (T4 has no bf16). 0.5B params fits easily.
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model.to("cuda")
print("Loaded", MODEL_NAME, "| params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct | params: 494.032768 M


## Tokenization with label masking

The model only learns from the **assistant answer**: prompt tokens get label `-100`, which the loss ignores.

In [6]:
from datasets import Dataset

def encode_example(ex):
    msgs = ex["messages"]
    prompt_text = tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)
    answer_text = msgs[2]["content"] + tokenizer.eos_token
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

encoded = [encode_example(r) for r in train_rows]
kept = [e for e in encoded if len(e["input_ids"]) <= MAX_LEN]
print(f"kept {len(kept)}/{len(encoded)} examples (dropped {len(encoded) - len(kept)} longer than {MAX_LEN} tokens)")
print("max length in kept set:", max(len(e["input_ids"]) for e in kept))

# sanity check: show which tokens are trained on
e = kept[0]
trained = [t for t, l in zip(e["input_ids"], e["labels"]) if l != -100]
print("\nTrained-on text:", repr(tokenizer.decode(trained)))

train_ds = Dataset.from_list(kept)

kept 3432/3432 examples (dropped 0 longer than 768 tokens)
max length in kept set: 312

Trained-on text: 'The priority of ticket 4177 is Low.<|im_end|>'


## Evaluation helpers

Metrics (greedy decoding, same test set for base and fine-tuned model):
- **lookup value match**: the answer contains the correct field value and the ticket ID
- **lookup exact match**: the answer equals the reference sentence (partly measures learned phrasing)
- **false refusal**: fallback answer given although the answer was in the context
- **out-of-scope refusal**: fallback answer given when the information does not exist

A second set of **paraphrased questions that do NOT come from the training templates** checks whether the model generalizes instead of memorizing templates.

In [7]:
import re

FIELD_LABEL = {"status": "Status", "priority": "Priority", "channel": "Channel",
               "type": "Ticket Type", "product": "Product", "subject": "Subject"}

def norm(s):
    return s.replace("\u2019", "'").lower().strip()

def gold_value(row):
    ctx = row["messages"][1]["content"]
    label = FIELD_LABEL[row["meta"]["field"]]
    m = re.search(rf"^{re.escape(label)}: (.*)$", ctx, re.M)
    return m.group(1).strip()

@torch.no_grad()
def generate_answers(model, rows, batch_size=16, max_new_tokens=48):
    model.eval()
    tokenizer.padding_side = "left"
    prompts = [tokenizer.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
               for r in rows]
    outputs = []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True,
                        add_special_tokens=False).to(model.device)
        with torch.autocast("cuda", dtype=torch.float16):
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        outputs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = "right"
    return [o.strip() for o in outputs]

def evaluate(model, rows):
    preds = generate_answers(model, rows)
    s = dict(lookup_n=0, value_ok=0, exact_ok=0, false_refusal=0, oos_n=0, oos_refused=0)
    records = []
    for r, pred in zip(rows, preds):
        meta, p = r["meta"], norm(pred)
        refused = "don't have enough information" in p
        if meta["kind"] == "lookup":
            s["lookup_n"] += 1
            s["value_ok"] += int(gold_value(r).lower() in p and meta["ticket_id"] in p)
            if meta.get("expected"):
                s["exact_ok"] += int(p.rstrip(".") == norm(meta["expected"]).rstrip("."))
            s["false_refusal"] += int(refused)
        else:
            s["oos_n"] += 1
            s["oos_refused"] += int(refused)
        records.append({"ticket_id": meta["ticket_id"], "kind": meta["kind"], "field": meta["field"],
                        "question": r["messages"][1]["content"].split("Question:\n")[-1].replace("\n\nAnswer:", ""),
                        "prediction": pred, "expected": meta.get("expected", "")})
    n_l, n_o = max(s["lookup_n"], 1), max(s["oos_n"], 1)
    metrics = {
        "lookup_n": s["lookup_n"], "oos_n": s["oos_n"],
        "lookup_value_match": round(s["value_ok"] / n_l, 3),
        "lookup_exact_match": round(s["exact_ok"] / n_l, 3),
        "false_refusal_rate": round(s["false_refusal"] / n_l, 3),
        "oos_refusal_rate": round(s["oos_refused"] / n_o, 3),
    }
    return metrics, records

# ---- paraphrased questions that are NOT in the training templates ----
PARAPHRASES = [
    ("lookup", "priority", "Please tell me how urgent ticket {tid} is."),
    ("lookup", "channel", "Which platform did the customer use to contact us for ticket {tid}?"),
    ("lookup", "product", "What item is ticket {tid} regarding?"),
    ("lookup", "status", "Give me the current state of ticket {tid}."),
    ("lookup", "type", "Which category does ticket {tid} fall under?"),
    ("out_of_scope", "none", "What is the customer's email address in ticket {tid}?"),
    ("out_of_scope", "none", "Which warehouse shipped the order in ticket {tid}?"),
    ("out_of_scope", "none", "What discount was offered in ticket {tid}?"),
    ("out_of_scope", "none", "What is the customer's age in ticket {tid}?"),
    ("out_of_scope", "none", "Which technician was assigned to ticket {tid}?"),
]

def build_paraphrase_rows(rows, n_tickets=100):
    seen, out = set(), []
    for r in rows:
        tid = r["meta"]["ticket_id"]
        if tid in seen:
            continue
        seen.add(tid)
        kind, field, tpl = PARAPHRASES[len(out) % len(PARAPHRASES)]
        ctx = r["messages"][1]["content"].split("\n\nQuestion:\n")[0]
        user = f"{ctx}\n\nQuestion:\n{tpl.format(tid=tid)}\n\nAnswer:"
        out.append({"messages": [r["messages"][0], {"role": "user", "content": user}],
                    "meta": {"ticket_id": tid, "kind": kind, "field": field, "expected": ""}})
        if len(out) >= n_tickets:
            break
    return out

para_rows = build_paraphrase_rows(test_rows)
print("paraphrase set:", len(para_rows))

paraphrase set: 100


In [8]:
# ===== BASE model (before fine-tuning) =====
base_test, base_test_records = evaluate(model, test_rows)
base_para, base_para_records = evaluate(model, para_rows)
print("BASE | template test :", base_test)
print("BASE | paraphrases    :", base_para)
print("\nExample base outputs:")
for rec in base_test_records[:3]:
    print("-", rec["question"], "->", rec["prediction"][:150])

BASE | template test : {'lookup_n': 400, 'oos_n': 52, 'lookup_value_match': 0.47, 'lookup_exact_match': 0.165, 'false_refusal_rate': 0.01, 'oos_refusal_rate': 0.096}
BASE | paraphrases    : {'lookup_n': 50, 'oos_n': 50, 'lookup_value_match': 0.5, 'lookup_exact_match': 0.0, 'false_refusal_rate': 0.02, 'oos_refusal_rate': 0.12}

Example base outputs:
- What subject does ticket 8 have? -> The subject of ticket 8 is "Battery life".
- How is ticket 8 classified by type? -> The ticket 8 is for a refund request related to Philips Hue Lights battery life.
- What is the serial number of the Philips Hue Lights in ticket 8? -> The serial number of the Philips Hue Lights in ticket 8 is 8.


In [10]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## LoRA training

In [12]:
import math
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
if not hasattr(model, "peft_config"):      # skip if LoRA was already attached
    model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACC))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(0.03 * total_steps))

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    report_to="none",
    seed=SEED,
)

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
trainer.train()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Step,Training Loss
20,0.114291
40,0.000336
60,0.000059
80,0.000048
100,0.000032
120,0.000030
140,0.000026
160,0.000023
180,0.000020
200,0.000019


TrainOutput(global_step=430, training_loss=0.005351306632808193, metrics={'train_runtime': 558.6718, 'train_samples_per_second': 12.286, 'train_steps_per_second': 0.77, 'total_flos': 3890055373946880.0, 'train_loss': 0.005351306632808193, 'epoch': 2.0})

In [13]:
# ===== Save adapter =====
ADAPTER_DIR = OUTPUT_DIR / "adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)

# ===== FINE-TUNED model =====
lora_test, lora_test_records = evaluate(model, test_rows)
lora_para, lora_para_records = evaluate(model, para_rows)

def show(title, base, lora):
    print(f"\n{title}")
    print(f"{'metric':<22}{'base':>8}{'LoRA':>8}")
    for k in ["lookup_value_match", "lookup_exact_match", "false_refusal_rate", "oos_refusal_rate"]:
        print(f"{k:<22}{base[k]:>8}{lora[k]:>8}")
    print(f"(n_lookup={lora['lookup_n']}, n_out_of_scope={lora['oos_n']})")

show("TEMPLATE TEST SET", base_test, lora_test)
show("PARAPHRASED QUESTIONS (not in training templates)", base_para, lora_para)

results = {"model": MODEL_NAME,
           "test": {"base": base_test, "lora": lora_test},
           "paraphrase": {"base": base_para, "lora": lora_para}}
with open(OUTPUT_DIR / "results_lora.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
with open(OUTPUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for tag, recs in [("base_test", base_test_records), ("lora_test", lora_test_records),
                      ("base_para", base_para_records), ("lora_para", lora_para_records)]:
        for rec in recs:
            f.write(json.dumps({"set": tag, **rec}, ensure_ascii=False) + "\n")
print("\nSaved results_lora.json and predictions.jsonl to", OUTPUT_DIR)

Saved adapter to /content/drive/MyDrive/lora_ticket_rag/adapter

TEMPLATE TEST SET
metric                    base    LoRA
lookup_value_match        0.47     1.0
lookup_exact_match       0.165     1.0
false_refusal_rate        0.01     0.0
oos_refusal_rate         0.096     1.0
(n_lookup=400, n_out_of_scope=52)

PARAPHRASED QUESTIONS (not in training templates)
metric                    base    LoRA
lookup_value_match         0.5    0.96
lookup_exact_match         0.0     0.0
false_refusal_rate        0.02     0.0
oos_refusal_rate          0.12     1.0
(n_lookup=50, n_out_of_scope=50)

Saved results_lora.json and predictions.jsonl to /content/drive/MyDrive/lora_ticket_rag


## Upload the adapter to the Hugging Face Hub

Requires the `HF_TOKEN` Colab Secret (write access). The token is read from Secrets, never written in the notebook.

In [14]:
from huggingface_hub import login, HfApi
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))
username = HfApi().whoami()["name"]
repo_id = f"{username}/{HF_REPO_NAME}"

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
print("Uploaded to: https://huggingface.co/" + repo_id)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 38.0kB / 35.2MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp7mo6ykd5/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Uploaded to: https://huggingface.co/mfatur/qwen2.5-0.5b-ticket-rag-lora


In [15]:
drive.flush_and_unmount()